# Example 2: Análise Setorial e Diversificação Geográfica

## Apêndice A, Seções A7, A12: Sector Analysis

Este notebook demonstra como o Credit Risk+ captura os efeitos de diversificação
através da análise setorial, onde fatores econômicos sistemáticos afetam grupos de obrigadores.

## 1. Conceito: Fatores Sistemáticos por Setor

Na realidade, defaults não são independentes. Eles são correlacionados através de fatores econômicos:
- **Setor bancário**: Afetado por taxa de juros, regulação
- **Setor imobiliário**: Afetado por preços de imóveis, taxas de desemprego
- **Setor tecnológico**: Afetado por ciclos de inovação

O modelo captura isso através de **setores** que compartilham volatilidade sistemática.

**Fórmula da PGF para múltiplos setores (Eq. 62-64):**
$$G(z) = \prod_{k=1}^n G_k(z)$$

Onde cada $G_k(z)$ é a PGF do setor $k$, e os setores são **independentes** entre si.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../creditriskplus')

from simple_model import calculate_loss_distribution_simple
from data import create_example_1a_portfolio

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Portfolio de Example 2: Três Setores Geográficos

Cada obrigador de Example 1A é alocado a exatamente UM setor:
- **EUA**: 10 obrigadores
- **Japão**: 8 obrigadores
- **Europa**: 7 obrigadores

In [ ]:
# Carrega exemplo 1A como base
portfolio_1a = create_example_1a_portfolio()

# Para Example 2, alocamos obrigadores a setores geográficos
# Simulamos: primeiros 10 ao EUA, próximos 8 ao Japão, resto à Europa
n_usa = 10
n_japan = 8
n_europe = 7

# Cria alocação setorial
usa_mask = np.arange(len(portfolio_1a)) < n_usa
japan_mask = (np.arange(len(portfolio_1a)) >= n_usa) & (np.arange(len(portfolio_1a)) < n_usa + n_japan)
europe_mask = np.arange(len(portfolio_1a)) >= n_usa + n_japan

print("=== ALOCAÇÃO SETORIAL - EXAMPLE 2 ===")
print(f"EUA:     {n_usa} obrigadores, Exposição: ${portfolio_1a[usa_mask]['exposure'].sum():,.0f}")
print(f"Japão:   {n_japan} obrigadores, Exposição: ${portfolio_1a[japan_mask]['exposure'].sum():,.0f}")
print(f"Europa:  {n_europe} obrigadores, Exposição: ${portfolio_1a[europe_mask]['exposure'].sum():,.0f}")
print(f"\nTotal:   25 obrigadores, Exposição: ${portfolio_1a['exposure'].sum():,.0f}")

## 3. Cálculo da Distribuição Agregada

A estratégia de cálculo:
1. Calcula distribuição para cada setor independentemente
2. Convolui as três distribuições setoriais
3. Resultado = distribuição agregada do portfólio

In [ ]:
# Calcula distribuição para cada setor
sectors = {
    'USA': (usa_mask, 'steelblue'),
    'Japan': (japan_mask, 'coral'),
    'Europe': (europe_mask, 'mediumseagreen')
}

sector_distributions = {}
sector_els = {}
sector_info = []

for sector_name, (mask, color) in sectors.items():
    sector_portfolio = portfolio_1a[mask]
    
    A, el = calculate_loss_distribution_simple(
        exposures=sector_portfolio['exposure'].values,
        mean_default_rates=sector_portfolio['mean_default_rate'].values,
        std_default_rates=sector_portfolio['std_default_rate'].values,
        recovery_rates=np.zeros(len(sector_portfolio)),
        unit_size=10000,
        max_units=15000
    )
    
    sector_distributions[sector_name] = A
    sector_els[sector_name] = el
    
    sector_info.append({
        'Setor': sector_name,
        'Obrigadores': len(sector_portfolio),
        'Exposição ($)': f"${sector_portfolio['exposure'].sum():,.0f}",
        'E[Loss] ($)': f"${el:,.0f}"
    })

df_sectors = pd.DataFrame(sector_info)
print("\n=== DISTRIBUIÇÕES SETORIAIS ===")
print(df_sectors.to_string(index=False))

total_el_example2 = sum(sector_els.values())
print(f"\nPerda Esperada Total: ${total_el_example2:,.0f}")

## 4. Convolução de Setores

In [ ]:
def convolve_dists(A1, A2):
    """Convolui duas distribuições."""
    max_len = min(len(A1) + len(A2) - 1, 30000)  # Limite para evitar memória
    result = np.zeros(max_len)
    for i in range(min(len(A1), max_len)):
        if A1[i] < 1e-15:
            continue
        for j in range(min(len(A2), max_len - i)):
            if A2[j] < 1e-15:
                continue
            if i + j < max_len:
                result[i + j] += A1[i] * A2[j]
    return result / np.sum(result)  # Normaliza

# Convolui setores
A_usa_japan = convolve_dists(sector_distributions['USA'], sector_distributions['Japan'])
A_example2 = convolve_dists(A_usa_japan, sector_distributions['Europe'])

# Validação
el_example2_check = np.sum(np.arange(len(A_example2)) * 10000 * A_example2)
print(f"\n=== VALIDAÇÃO APÓS CONVOLUÇÃO ===")
print(f"Soma de E[L] por setor: ${total_el_example2:,.0f}")
print(f"E[L] após convolução:   ${el_example2_check:,.0f}")
print(f"Match: {np.isclose(total_el_example2, el_example2_check, rtol=0.01)}")

## 5. Impacto da Diversificação Setorial

Comparamos Example 2 (setorial) com Example 1A (sem setorialização).

In [ ]:
# Carrega Example 1A agregado (todos em um setor)
A_1a, el_1a = calculate_loss_distribution_simple(
    exposures=portfolio_1a['exposure'].values,
    mean_default_rates=portfolio_1a['mean_default_rate'].values,
    std_default_rates=portfolio_1a['std_default_rate'].values,
    recovery_rates=np.zeros(len(portfolio_1a)),
    unit_size=10000,
    max_units=15000
)

# Calcula CDFs
cdf_1a = np.cumsum(A_1a)
cdf_ex2 = np.cumsum(A_example2)

# Percentis
percentiles = [50, 75, 95, 99, 99.5]
results = []

for p in percentiles:
    idx_1a = np.searchsorted(cdf_1a, p/100.0)
    loss_1a = idx_1a * 10000
    
    idx_ex2 = np.searchsorted(cdf_ex2, p/100.0)
    loss_ex2 = idx_ex2 * 10000
    
    reduction = loss_1a - loss_ex2
    
    results.append({
        'Percentil': f"{p:.1f}%",
        'Example 1A ($)': f"{loss_1a:,.0f}",
        'Example 2 ($)': f"{loss_ex2:,.0f}",
        'Redução ($)': f"{reduction:,.0f}",
        'Redução %': f"{reduction/loss_1a*100:>5.1f}%"
    })

df_comp = pd.DataFrame(results)
print("\n=== IMPACTO DA DIVERSIFICAÇÃO SETORIAL ===")
print(df_comp.to_string(index=False))

print(f"\nInterpretação:")
print(f"- E[Loss] é igual (mesmos obrigadores)")
print(f"- Percentis altos são MENORES em Example 2")
print(f"- Razão: Independência entre setores reduz correlação extrema")
print(f"- Efeito é maior nas caudas (99%, 99.5%)")

## 6. Visualização Comparativa

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

unit_size = 10000
max_idx = 10000
loss_vals = np.arange(max_idx) * unit_size

# PMF setorial (Setor USA)
ax = axes[0, 0]
ax.plot(loss_vals, sector_distributions['USA'][:max_idx], linewidth=2, label='USA', color='steelblue')
ax.set_xlabel('Perda ($)', fontsize=10)
ax.set_ylabel('PMF', fontsize=10)
ax.set_title('Setor USA - PMF', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# PMF setorial (Setor Japão)
ax = axes[0, 1]
ax.plot(loss_vals, sector_distributions['Japan'][:max_idx], linewidth=2, label='Japan', color='coral')
ax.set_xlabel('Perda ($)', fontsize=10)
ax.set_ylabel('PMF', fontsize=10)
ax.set_title('Setor Japão - PMF', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# CDF Agregada
ax = axes[1, 0]
ax.plot(loss_vals, cdf_1a[:max_idx], linewidth=2.5, label='Example 1A (Sem Setores)', color='black')
ax.plot(loss_vals, cdf_ex2[:max_idx], linewidth=2.5, label='Example 2 (Com Setores)', color='darkgreen')
ax.axhline(0.99, color='red', linestyle='--', alpha=0.3)
ax.set_xlabel('Perda ($)', fontsize=10)
ax.set_ylabel('CDF', fontsize=10)
ax.set_title('Impacto da Diversificação Setorial', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Capital econômico por setor
ax = axes[1, 1]
sectors_list = list(sectors.keys())
els_list = [sector_els[s] for s in sectors_list]
colors_list = [sectors[s][1] for s in sectors_list]
ax.bar(sectors_list, els_list, color=colors_list, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Perda Esperada ($)', fontsize=10)
ax.set_title('Perda Esperada por Setor', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Conclusão: Poder da Análise Setorial

O modelo Credit Risk+ com setores permite:

1. **Capturar Correlação Realista**: Obrigadores no mesmo setor são mais correlacionados
2. **Quantificar Diversificação**: Mostrar benefício exato de adicionar setores
3. **Análise de Stress**: Estressar um setor mantendo outros constantes
4. **Pricing**: Calcular spread por setor baseado em seu risco sistemático

**Resultado numérico**: Example 2 (com setores) tem **menor risco de cauda** que Example 1A (sem setores),
mostrando o valor da diversificação geográfica.